# IBKR API notebook

Connection

In [ ]:
from ib_async import *
from datetime import datetime
util.startLoop()

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=14)

<IB connected to 127.0.0.1:7497 clientId=14>

Error 162, reqId 6: Message d'erreur Service Donn\u00e9es de March\u00e9 Historiques:API historical data query cancelled: 6, contract: Index(symbol='NDX', exchange='NASDAQ', currency='USD')
Error 162, reqId 7: Message d'erreur Service Donn\u00e9es de March\u00e9 Historiques:API historical data query cancelled: 7, contract: Index(symbol='NDX', exchange='NASDAQ', currency='USD')


### Request Historical data

Choose your contract

In [ ]:
contract = Stock('AAPL', 'SMART', 'USD')

In [ ]:
contract = Index('NDX', 'NASDAQ', 'USD')

In [ ]:
contract = Forex('EURUSD')

Check first data date available

In [ ]:
timestamp = ib.reqHeadTimeStamp(contract, whatToShow='TRADES', useRTH=True)
formatted_time = timestamp.strftime(%B %d, %Y, %H:%M)
print(f"First date of data available: {formatted_time}")

datetime.datetime(2004, 3, 4, 14, 30)

Request historical data function

In [ ]:
bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr='1 y',
        barSizeSetting='1 min',
        whatToShow='TRADES',
        useRTH=True,
        formatDate=1,
        timeout = 600)

In [18]:
bars[0]

BarData(date=datetime.datetime(2024, 4, 3, 9, 30, tzinfo=zoneinfo.ZoneInfo(key='US/Eastern')), open=18054.45, high=18054.45, low=18045.42, close=18050.28, volume=0.0, average=0.0, barCount=45)

Convert the list of bars to a data frame and print the first and last rows:

In [19]:
df = util.df(bars)

display(df.head())
display(df.tail())

,date,open,high,low,close,volume,average,barCount
0,2024-04-03 09:30:00-04:00,18054.45,18054.45,18045.42,18050.28,0.0,0.0,45
1,2024-04-03 09:31:00-04:00,18049.97,18059.20,18046.97,18055.07,0.0,0.0,59
2,2024-04-03 09:32:00-04:00,18054.71,18067.66,18046.89,18063.90,0.0,0.0,60
3,2024-04-03 09:33:00-04:00,18062.20,18063.30,18056.83,18061.77,0.0,0.0,60
4,2024-04-03 09:34:00-04:00,18062.30,18071.19,18062.30,18068.13,0.0,0.0,60


,date,open,high,low,close,volume,average,barCount
97167,2025-04-02 12:57:00-04:00,19618.34,19626.17,19617.09,19624.62,0.0,0.0,59
97168,2025-04-02 12:58:00-04:00,19624.46,19630.57,19623.02,19624.60,0.0,0.0,59
97169,2025-04-02 12:59:00-04:00,19623.79,19641.14,19623.37,19638.63,0.0,0.0,60
97170,2025-04-02 13:00:00-04:00,19638.43,19649.23,19637.12,19648.26,0.0,0.0,59
97171,2025-04-02 13:01:00-04:00,19646.54,19649.26,19638.64,19646.19,0.0,0.0,59


Save your pulled data in a dataframe

In [ ]:
df.to_parquet('bars.parquet', index=True, compression=None,)

Instruct the notebook to draw plot graphics inline:

In [ ]:
%matplotlib inline

Plot the close data

In [ ]:
df.plot(y='close');

There is also a utility function to plot bars as a candlestick plot. It can accept either a DataFrame or a list of bars. Here it will print the last 100 bars:

In [ ]:
util.barplot(bars[-100:], title=contract.symbol);

## Historical data with realtime updates

A new feature of the API is to get live updates for historical bars. This is done by setting `endDateTime` to an empty string and the `keepUpToDate` parameter to `True`.

Let's get some bars with an keepUpToDate subscription:

In [ ]:

bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr='900 S',
        barSizeSetting='10 secs',
        whatToShow='MIDPOINT',
        useRTH=True,
        formatDate=1,
        keepUpToDate=True)

Replot for every change of the last bar:

In [ ]:
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

def onBarUpdate(bars, hasNewBar):
    plt.close()
    plot = util.barplot(bars)
    clear_output(wait=True)
    display(plot)

bars.updateEvent += onBarUpdate

ib.sleep(10)
ib.cancelHistoricalData(bars)

Realtime bars
------------------

With ``reqRealTimeBars`` a subscription is started that sends a new bar every 5 seconds.

First we'll set up a event handler for bar updates:

In [ ]:
def onBarUpdate(bars, hasNewBar):
    print(bars[-1])

Then do the real request and connect the event handler,

In [ ]:
bars = ib.reqRealTimeBars(contract, 5, 'MIDPOINT', False)
bars.updateEvent += onBarUpdate

let it run for half a minute and then cancel the realtime bars.

In [ ]:
ib.sleep(30)
ib.cancelRealTimeBars(bars)

The advantage of reqRealTimeBars is that it behaves more robust when the connection to the IB server farms is interrupted. After the connection is restored, the bars from during the network outage will be backfilled and the live bars will resume.

reqHistoricalData + keepUpToDate will, at the moment of writing, leave the whole API inoperable after a network interruption.

### Request historical market news

In [ ]:
ib.reqHistoricalNews()
#ib.reqHistoricalNewsAsync()

In [ ]:
ib.disconnect()